In [1]:
# Pregunta analítica: ¿qué política de contacto debe aprobar la responsable comercial antes de cada campaña de depósito, sin exceder la capacidad ni el límite de exposición?

from pathlib import Path
import sys

import pandas as pd

workspace = Path.cwd().resolve()
activity_dir = next(
    directory
    for directory in [workspace, *workspace.parents]
    if (directory / 'data' / 'policy_options.csv').is_file()
)
sys.path.insert(0, str(activity_dir / 'src'))

from main import evaluate_policies, make_decision_brief, make_policy_contract


In [2]:
# La evidencia predictiva es un insumo; todavía no determina la acción.

options = pd.read_csv(activity_dir / 'data' / 'policy_options.csv')
options


,policy_id,policy,contacts,expected_conversion,contact_cost,net_value_per_conversion,high_contact_share,capacity_limit,max_high_contact_share
0,P0,no_contactar,0,0.000,0,180,0.00,1000,0.35
1,P1,solo_clientes_con_probabilidad_alta,600,0.180,3,180,0.20,1000,0.35
2,P2,ampliar_a_probabilidad_media,1000,0.115,3,180,0.32,1000,0.35
3,P3,contactar_toda_la_lista,1250,0.100,3,180,0.42,1000,0.35


In [3]:
# Una política solo puede recomendar acciones factibles bajo sus restricciones.

evaluated = evaluate_policies(options)
evaluated.loc[:, ['policy', 'expected_conversions', 'expected_net_value', 'capacity_ok', 'exposure_ok', 'feasible']]


,policy,expected_conversions,expected_net_value,capacity_ok,exposure_ok,feasible
0,no_contactar,0.0,0.0,True,True,True
1,solo_clientes_con_probabilidad_alta,108.0,17640.0,True,True,True
2,ampliar_a_probabilidad_media,115.0,17700.0,True,True,True
3,contactar_toda_la_lista,125.0,18750.0,False,False,False


In [4]:
# La salida incluye la acción y el contrato que permite operarla y revisarla.

brief = make_decision_brief(options)
policy_contract = make_policy_contract(brief)

policy_contract.T


,0
policy_id,P2
policy,ampliar_a_probabilidad_media
contacts,1000
expected_conversions,115.0
expected_net_value,17700.0
capacity_ok,True
exposure_ok,True
decision_owner,Responsable comercial
review_trigger,Revisar si la conversión observada cae por deb...
decision_cadence,Antes de cada campaña de depósito


In [5]:
brief.to_csv(activity_dir / 'submission' / 'decision_brief.csv', index=False)
policy_contract.to_csv(activity_dir / 'submission' / 'policy_contract.csv', index=False)
